# Killer motility: target contacts and kills

**Run All** to load a completed local sweep and plot it. The notebook prints the folder and simulation count before drawing the figure.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import Normalize

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "paper/section_2/migration").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the MantiShrimp checkout.")
MIGRATION_DIR = PROJECT_ROOT / "paper/section_2/migration"

PARAMETERS = ["alpha", "sigma_mu", "sigma_r"]
SWEEP_FILES = ("settings.json", "runs.csv.gz", "summary.csv")

import scienceplots
plt.style.use(["science"])
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix", "text.usetex": False,
    # "font.size": 13, "axes.labelsize": 14, "axes.titlesize": 15,
    # "figure.titlesize": 16, "figure.labelsize": 14,
    # "xtick.labelsize": 13, "ytick.labelsize": 13,
    # "xtick.major.size": 7, "ytick.major.size": 7,
    # "xtick.major.width": 1.25, "ytick.major.width": 1.25,
    "xtick.minor.visible": False, "ytick.minor.visible": False,
    "xtick.top": False, "ytick.right": False,
    "axes.linewidth": 1.1, 
    # "axes.spines.top": False, "axes.spines.right": False,
    # "legend.frameon": False, "svg.fonttype": "none", "figure.dpi": 120,
})

In [ ]:
def select_results_dir(requested, available):
    """Honour an explicit path; otherwise prefer the full sweep and explain the choice."""
    if requested is not None:
        return MIGRATION_DIR / Path(requested).expanduser()
    if not available:
        raise FileNotFoundError("No local sweep tables found. Download a completed run or set SWEEP_TO_LOAD to its data folder.")
    full = []
    for folder in available:
        cfg = json.loads((folder / "settings.json").read_text())
        if all(len(cfg["grids"][p]) == 10 for p in PARAMETERS) and len(cfg["seeds"]) == 5:
            full.append(folder)
    choices = full or available
    if len(choices) != 1:
        raise ValueError("More than one sweep matches. Set SWEEP_TO_LOAD to one of the listed data folders.")
    if not full:
        print(f"The 5,000-simulation sweep is not available locally. Using the only local sweep: {choices[0].parent.name}.")
    return choices[0]


def load_results(results_dir):
    """Read the saved grid and check the per-simulation and summary tables."""
    results_dir = Path(results_dir)
    missing = [name for name in SWEEP_FILES if not (results_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing sweep results in {results_dir}: {', '.join(missing)}. "
            "Download this sweep, or set SWEEP_TO_LOAD = None to use a local run."
        )
    settings = json.loads((results_dir / "settings.json").read_text())
    runs = pd.read_csv(results_dir / "runs.csv.gz", float_precision="round_trip")
    summary = pd.read_csv(results_dir / "summary.csv", float_precision="round_trip")
    run_keys = PARAMETERS + ["seed"]
    metrics = ["unique_contacts_per_killer", "kills_per_killer"]
    for frame, columns in (
        (runs, run_keys + metrics),
        (summary, PARAMETERS + [f"mean_{metric}" for metric in metrics] + ["n_runs"]),
    ):
        if frame.empty or not set(columns).issubset(frame.columns):
            raise ValueError(f"Results must contain nonempty columns: {', '.join(columns)}")
    if runs.duplicated(run_keys).any() or summary.duplicated(PARAMETERS).any():
        raise ValueError("Duplicate parameter/seed rows in the saved results.")
    expected = pd.MultiIndex.from_product(
        [settings["grids"][name] for name in PARAMETERS] + [settings["seeds"]],
        names=run_keys,
    )
    actual = pd.MultiIndex.from_frame(runs[run_keys])
    if len(expected.difference(actual)) or len(actual.difference(expected)):
        raise ValueError(f"Incomplete or mismatched grid: {len(runs)} rows; expected {len(expected)}.")
    expected_conditions = expected.droplevel("seed").unique()
    actual_conditions = pd.MultiIndex.from_frame(summary[PARAMETERS])
    if (len(expected_conditions.difference(actual_conditions))
            or len(actual_conditions.difference(expected_conditions))
            or not summary["n_runs"].eq(len(settings["seeds"])).all()):
        raise ValueError("The summary is incomplete or does not match settings.json.")
    for metric in metrics:
        values = runs[metric]
        means = runs.groupby(PARAMETERS)[metric].mean().sort_index()
        saved_means = summary.set_index(PARAMETERS)[f"mean_{metric}"].sort_index()
        if (not np.isfinite(values).all() or (values < 0).any()
                or not np.isfinite(saved_means).all() or (saved_means < 0).any()
                or not np.allclose(means.to_numpy(), saved_means.to_numpy())):
            raise ValueError(f"The summary for {metric} does not match valid per-run results.")
    return settings, runs, summary

## 1. Import a completed sweep

Leave `SWEEP_TO_LOAD = None` and run the cell below. It checks `mig_sweep/runs/` each time and prints the folder and simulation count it loads. It prefers the **5,000-simulation sweep** (10 × 10 × 10 settings, five seeds). If that sweep is absent, it uses the only other local sweep. If several folders match, it asks you to choose one.

To plot a specific sweep, paste its listed `data` path in quotes in place of `None`, then choose **Run All**. An explicit path must contain `settings.json`, `runs.csv.gz` and `summary.csv`; the notebook will stop if any are missing.

In [ ]:
SWEEP_TO_LOAD = None  # Or paste a listed data folder here in quotes.

local_data = sorted(d for d in (MIGRATION_DIR / "mig_sweep/runs").glob("run_*/data")
                    if all((d / f).is_file() for f in SWEEP_FILES))
print("Downloaded data folders (relative to MIGRATION_DIR):")
print("\n".join(f"  {d.relative_to(MIGRATION_DIR)}" for d in local_data) or "  None found.")

RESULTS_DIR = select_results_dir(SWEEP_TO_LOAD, local_data)
settings, runs, summary = load_results(RESULTS_DIR)
sweep_folder = RESULTS_DIR.parent if RESULTS_DIR.name == "data" else RESULTS_DIR
FIGURE_DIR = MIGRATION_DIR / "figures" / sweep_folder.name

print(f"Loaded: {RESULTS_DIR}")
print(f"{len(runs):,} simulations = {len(summary):,} parameter combinations × {len(settings['seeds'])} seeds")
if settings["smoke_test"]:
    print("Smoke test: this run checks that the simulation executes.")
elif settings.get("pilot", False):
    print("Pilot: only the grid extremes were sampled.")

preview = summary[PARAMETERS + ["mean_unique_contacts_per_killer", "mean_kills_per_killer", "n_runs"]].head()
display(preview.rename(columns={"mean_unique_contacts_per_killer": "Distinct targets per killer",
                                "mean_kills_per_killer": "Kills per killer", "n_runs": "Seeds"}))

## 2. Plot contacts and kills

Each row holds translational noise ($\sigma_r$) fixed. Within a panel, polarity decay ($\alpha$) runs horizontally and polarity noise ($\sigma_\mu$) vertically. Each tile averages first across killers in a simulation and then across replicate seeds.

The blue panels count distinct targets contacted per killer; a killer meeting the same target twice counts that target once. The red panels show kills per killer. Each column uses one colour scale across all its rows.

In [ ]:
def plot_heatmaps(summary, settings, smoke_test=False, pilot=False):
    sigma_r_values = sorted(settings["grids"]["sigma_r"])
    if not sigma_r_values:
        raise ValueError("The sigma_r grid is empty.")

    metrics = [
        ("mean_unique_contacts_per_killer", "Unique target contacts per killer", "Blues"),
        ("mean_kills_per_killer", "Kills per killer", "Reds"),
    ]
    matrices = [
        [heatmap_slice(summary, settings, sigma_r, metric)
            for metric, _, _ in metrics]
        for sigma_r in sigma_r_values
    ]
    norms = [
        Normalize(
            vmin=0,
            vmax=max(max(row[col].to_numpy().max() for row in matrices), 1e-12),
        )
        for col in range(2)
    ]
    
    fig, axes = plt.subplots(
            len(sigma_r_values), 2,
            figsize=(12, 4.2 * len(sigma_r_values)),
            squeeze=False,
            layout="constrained",
        )
    for col, (_, title, cmap) in enumerate(metrics):
        for row, sigma_r in enumerate(sigma_r_values):
                ax = axes[row, col]
                matrix = matrices[row][col]
                values = matrix.to_numpy()
                im = ax.imshow(
                    values, origin="lower", aspect="auto",
                    cmap=cmap, norm=norms[col], interpolation="nearest",
                )
                ax.set_xticks(
                    range(len(matrix.columns)), [f"{v:g}" for v in matrix.columns]
                )
                ax.set_yticks(
                    range(len(matrix.index)), [f"{v:g}" for v in matrix.index]
                )
                ax.set(
                    xlabel=r"Killer polarity decay, $\alpha$",
                    ylabel=r"Polarity noise, $\sigma_\mu$",
                    title=rf"{title} — $\sigma_r={sigma_r:g}$",
                )
                ax.grid(False)
                for (i, j), value in np.ndenumerate(values):
                    ax.text(
                        j, i, f"{value:.2f}",
                        ha="center", va="center", fontsize=9,
                        color="white" if norms[col](value) > 0.6 else "#172a3a",
                    )
        fig.colorbar(
                im, ax=axes[:, col].tolist(), shrink=0.88, label=title
            )

        fig.suptitle(
            "Killer motility: target contacts and kills"
            + (" (smoke test)" if smoke_test else " (pilot)" if pilot else "")
            + f"\n{int(summary['n_runs'].sum()):,} simulations; {len(summary):,} parameter combinations"
        )

    assert axes.shape == (len(sigma_r_values), 2)
    return fig


def heatmap_slice(frame, settings, sigma_r, metric):
    """Return a sigma_mu × alpha grid at one sigma_r."""
    selected = frame.loc[frame["sigma_r"].eq(sigma_r)]
    if selected.empty:
        raise ValueError(f"sigma_r={sigma_r:g} was not swept.")
    matrix = selected.pivot(index="sigma_mu", columns="alpha", values=metric)
    matrix = matrix.reindex(
        index=sorted(settings["grids"]["sigma_mu"]),
        columns=sorted(settings["grids"]["alpha"]),
    )
    if not np.isfinite(matrix.to_numpy()).all():
        raise ValueError(f"Missing or invalid {metric} values at sigma_r={sigma_r:g}.")
    return matrix

def save_figure(fig, output_dir):
    """Export the figure in PDF, SVG and PNG formats."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = "motility_contacts_and_kills_by_sigma_r"
    with plt.rc_context({"svg.fonttype": "none", "pdf.fonttype": 42}):
        for extension in ("pdf", "svg"):
            path = output_dir / f"{stem}.{extension}"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            print(f"Saved {path.resolve()}", flush=True)
    return stem

fig = plot_heatmaps(
    summary, 
    settings,
    smoke_test=settings["smoke_test"],
    pilot=settings.get("pilot", False),
)
# stem = save_figure(fig, FIGURE_DIR)
plt.show()

In [ ]:
sigma_r = 5
# metric = "mean_unique_contacts_per_killer"
metric = "mean_synapses"
# metric = "mean_kills_per_killer"
# metric = "mean_kills_per_killer"
matrix = heatmap_slice(summary, settings, sigma_r, metric)
matrix = matrix / settings["base_config"]["n_killers"]
norm = Normalize(vmin=0, vmax=max(matrix.to_numpy().max(), 1e-12))

fig, ax = plt.subplots(figsize=(7, 6), layout="constrained", dpi = 280)
ax.set_box_aspect(1)

im = ax.imshow(
    matrix.to_numpy(), origin="lower", aspect="auto",
    cmap="RdBu_r", norm=norm, interpolation="nearest",
)
#coolwarm
#RdBu_r

ax.set_xticks(range(len(matrix.columns)), [f"{v:g}" for v in matrix.columns])
ax.set_yticks(range(len(matrix.index)), [f"{v:g}" for v in matrix.index])
ax.set(
    xlabel=r"Polarity decay of killer cells $\alpha$",
    ylabel=r"Polarity noise of killer cells $\sigma_\mu$",
    title=rf"$\sigma_r={sigma_r:g}$",
)
ax.grid(False)

# for (i, j), value in np.ndenumerate(matrix.to_numpy()):
#     ax.text(
#         j, i, f"{value:.2f}", ha="center", va="center", fontsize=8,
#         color="white" if norm(value) > 0.6 else "#172a3a",
#     )
plt.rcParams.update({
    "font.size": 12,          # Default text size
    "axes.labelsize": 22,     # x/y labels and colourbar label
    "axes.titlesize": 24,     # Heatmap title
    "figure.titlesize": 24,   # Figure title, if using fig.suptitle()
    "xtick.labelsize": 16,    # x-axis tick labels
    "ytick.labelsize": 16,    # y-axis and vertical colourbar tick labels
})

fig.colorbar(im, ax=ax, label="Synapse formed per killer cell")

for extension in ("pdf", "svg"):
    path = FIGURE_DIR / f"motility_synapses_by_sigma_r.{extension}"
    fig.savefig(path, dpi=300, bbox_inches="tight", transparent=True)
    print(f"Saved {path.resolve()}", flush=True)
plt.show()

In [ ]:
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import to_rgba

sigma_r = 5

colour_table = pd.DataFrame([
    ("A", 0.75, 0.6, "Blue",   "#0068A9"),
    ("B", 1.75, 1.0, "Orange", "#E9630E"),
    ("C", 2.50, 1.2, "Purple", "#9E4589"),
], columns=["Panel", "alpha", "sigma_mu", "Colour", "Hex"])

display(colour_table)

killers = pd.read_csv(
    RESULTS_DIR / "killers.csv.gz",
    float_precision="round_trip",
)

expected_n = settings["base_config"]["n_killers"] * len(settings["seeds"])
pooled_counts = []

for row in colour_table.itertuples(index=False):
    selected = killers.loc[
        np.isclose(killers["alpha"], row.alpha)
        & np.isclose(killers["sigma_mu"], row.sigma_mu)
        & np.isclose(killers["sigma_r"], sigma_r)
    ]

    assert len(selected) == expected_n and not selected.duplicated(
        ["seed", "killer_id"]
    ).any(), f"Missing or duplicate cells for panel {row.Panel}"

    pooled_counts.append(selected["synapses"].to_numpy())

max_count = int(max(values.max() for values in pooled_counts))
x = np.arange(max_count + 1)
bins = np.arange(-0.5, max_count + 1.5, 1)

plt.rcParams.update({
    "font.size": 20,
    "axes.labelsize": 26,
    "axes.titlesize": 20,
    "figure.titlesize": 22,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 18,
})

fig, axes = plt.subplots(
    3, 1,
    figsize=(9, 7),
    sharex=True,
    sharey=True,
    layout="constrained",
    dpi=280,
)

for ax, row, values in zip(
    axes, colour_table.itertuples(index=False), pooled_counts
):
    # Unit-width bins retain the density values from the previous histogram.
    density, _ = np.histogram(values, bins=bins, density=True)

    ax.bar(
        x, density,
        width=0.85,
        color=to_rgba(row.Hex, alpha=0.6),
        edgecolor="black",
        linewidth=1.5,
    )

    ax.set(
        ylabel="Density",
        xlim=(-0.5, max_count + 0.5),
    )
    ax.set_title(
        rf"{row.Panel}: $\alpha={row.alpha:g},\quad \sigma_\mu={row.sigma_mu:g}$",
        color=row.Hex,
        y=0.95,
        va="top",
        pad=0,
    )
    ax.text(
        0.98, 0.94, f"n = {len(values)}",
        transform=ax.transAxes,
        ha="right", va="top", fontsize=18,
    )
    ax.grid(False)
    ax.set_xlim(-0.8, max_count + 0.6)

axes[-1].set_xlabel("Synapse formations per killer cell")
axes[-1].xaxis.set_major_locator(MaxNLocator(integer=True))
fig.suptitle(rf"$\sigma_r={sigma_r:g}$")

for extension in ("pdf", "svg"):
    path = FIGURE_DIR / f"synapse_distribution_three.{extension}"
    fig.savefig(path, dpi=300, bbox_inches="tight", transparent=True)
    print(f"Saved {path.resolve()}", flush=True)
plt.show()

# for ax, row, values in zip(
#     axes,
#     colour_table.iloc[::-1].itertuples(index=False),
#     reversed(pooled_counts),
# ):
#     density, _ = np.histogram(values, bins=bins, density=True)

#     ax.bar(
#         x, density,
#         width=0.85,
#         color=to_rgba(row.Hex, alpha=0.6),
#         edgecolor="black",
#         linewidth=1.5,
#     )

#     ax.set(
#         ylabel="Density",
#         xlim=(-0.8, max_count + 0.6),
#     )

#     ax.set_title(
#         rf"$\alpha={row.alpha:g},\quad \sigma_\mu={row.sigma_mu:g}$",
#         color=row.Hex,
#         y=0.95,
#         va="top",
#         pad=0,
#     )

#     ax.text(
#         0.98, 0.94, f"n = {len(values)}",
#         transform=ax.transAxes,
#         ha="right", va="top", fontsize=18,
#     )
#     ax.grid(False)

In [ ]:
NATURE_PALETTES = {
    "Grey":   ("#E2E2EA", "#C1C7D5", "#8E99AB", "#647086", "#3F4C62", "#212C3E"),
    "Blue":   ("#C1E4FA", "#92C4E9", "#4C8FCB", "#0068A9", "#004586", "#002753"),
    "Teal":   ("#C5E3EB", "#8CCCCE", "#42B4B5", "#008F99", "#015D6E", "#003245"),
    "Purple": ("#E8D0E6", "#CDA0CB", "#B271AB", "#9E4589", "#6D2769", "#3F1947"),
    "Red":    ("#F9C9C7", "#EA9A9D", "#D95D5B", "#C23637", "#912321", "#691816"),
    "Yellow": ("#FFEFBB", "#F5D77F", "#E8C047", "#C59527", "#926C18", "#614B14"),
    "Orange": ("#FDDBB5", "#FBB874", "#F48F3D", "#E9630E", "#AE460A", "#792C03"),
}

sigma_r = 5
conditions = [
    (2.50, 1.2, "Purple"),
    (1.75, 1.0, "Orange"),
    (0.75, 0.6, "Blue"),
]
seeds = sorted(settings["seeds"])

plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "figure.titlesize": 22,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10,
})

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.5),
    sharex=True,
    sharey=True,
    layout="constrained",
    dpi=280,
)

for ax, (alpha, sigma_mu, family) in zip(axes, conditions):
    panel_runs = runs.loc[
        np.isclose(runs["alpha"], alpha)
        & np.isclose(runs["sigma_mu"], sigma_mu)
        & np.isclose(runs["sigma_r"], sigma_r)
    ].sort_values("seed")

    assert panel_runs["seed"].tolist() == seeds, "Missing or duplicate seed runs."

    # Skip the palest shade to keep all five curves visible.
    shades = NATURE_PALETTES[family][1:]
    assert len(seeds) <= len(shades), "More seeds than available colour shades."

    for run, colour in zip(panel_runs.itertuples(index=False), shades):
        events = pd.read_csv(
            RESULTS_DIR / run.events_file,
            float_precision="round_trip",
        )
        assert len(events) == run.n_events, (
            f"Incomplete event log for seed {run.seed}."
        )

        times = (
            events.loc[events["event"].eq("synapse_formed"), "time"]
            .sort_values()
            .to_numpy()
        )

        assert len(times) == run.synapses, (
            f"Synapse count mismatch for seed {run.seed}."
        )
        assert np.isfinite(times).all() and (
            (times >= 0) & (times <= run.obs_time)
        ).all()

        # Extend each curve from time zero to the end of the simulation.
        t = np.r_[0.0, times, run.obs_time]
        cumulative = (
            np.r_[0, np.arange(1, len(times) + 1), len(times)]
            / run.n_killers
        )

        ax.step(
            t, cumulative,
            where="post",
            color=colour,
            linewidth=1.8,
            label=f"Seed {run.seed}",
        )

    ax.set_title(
        rf"$\alpha={alpha:g},\quad \sigma_\mu={sigma_mu:g}$",
        color=NATURE_PALETTES[family][3],
        y=0.95,
        va="top",
        pad=0,
    )
    ax.set(
        xlabel="Simulation time",
        xlim=(0, panel_runs["obs_time"].max()),
    )
    ax.legend(loc="lower right", frameon=False)
    ax.grid(False)

axes[0].set_ylabel("Cumulative synapse formations\nper killer cell")
axes[0].set_ylim(bottom=0)
fig.suptitle(rf"$\sigma_r={sigma_r:g}$")

plt.show()

# Validated all 15 curves against saved counts; notebook unchanged.

In [ ]:
from matplotlib.ticker import MaxNLocator

sigma_r = 5
conditions = [
    (2.50, 1.2, "Purple"),
    (1.75, 1.0, "Orange"),
    (0.75, 0.6, "Blue"),
]
seeds = sorted(settings["seeds"])

plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "figure.titlesize": 22,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10,
})

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5),
    sharex=True,
    sharey=True,
    layout="constrained",
    dpi=280,
)

for ax, (alpha, sigma_mu, family) in zip(axes, conditions):
    panel_runs = runs.loc[
        np.isclose(runs["alpha"], alpha)
        & np.isclose(runs["sigma_mu"], sigma_mu)
        & np.isclose(runs["sigma_r"], sigma_r)
    ].sort_values("seed")

    assert panel_runs["seed"].tolist() == seeds, "Missing or duplicate seed runs."

    shades = NATURE_PALETTES[family][1:]
    assert len(seeds) <= len(shades), "More seeds than available colour shades."

    for run, colour in zip(panel_runs.itertuples(index=False), shades):
        events = pd.read_csv(
            RESULTS_DIR / run.events_file,
            float_precision="round_trip",
        )
        assert len(events) == run.n_events, "Incomplete event log."

        syn = events.loc[
            events["event"].isin(["synapse_formed", "synapse_ended"]),
            ["time", "event"],
        ].copy()

        assert syn["event"].eq("synapse_formed").sum() == run.synapses
        assert np.isfinite(syn["time"]).all() and (
            syn["time"].between(0, run.obs_time).all()
        )

        syn["change"] = syn["event"].map({
            "synapse_formed": 1,
            "synapse_ended": -1,
        })

        # Combine simultaneous events before updating the active bond count.
        active = (
            syn.groupby("time", sort=True)["change"]
            .sum()
            .cumsum()
        )
        assert active.ge(0).all(), "Unmatched synapse-ending events."

        t = np.r_[0.0, active.index.to_numpy(), run.obs_time]
        levels = np.r_[0, active.to_numpy()]

        ax.step(
            t,
            np.r_[levels, levels[-1]],
            where="post",
            color=colour,
            linewidth=1.2,
            label=f"Seed {run.seed}",
        )

    ax.set_title(
        rf"$\alpha={alpha:g},\quad \sigma_\mu={sigma_mu:g}$",
        color=NATURE_PALETTES[family][3],
        y=0.95,
        va="top",
        pad=0,
    )
    ax.set(
        xlabel="Simulation time",
        xlim=(0, panel_runs["obs_time"].max()),
    )
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),
        ncol=3,
        frameon=False,
    )
    ax.grid(False)

axes[0].set_ylabel("Total active conjugations")
axes[0].margins(y=0.20)
axes[0].set_ylim(bottom=0)
axes[0].yaxis.set_major_locator(MaxNLocator(integer=True))
fig.suptitle(rf"$\sigma_r={sigma_r:g}$")

plt.show()

# Validated all 15 curves against individual bond events; notebook unchanged.